In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
FILE_PATH = '/kaggle/input/datasets/yasserh/nyc-taxi-trip-duration/NYC.csv'

MIN_TRIP_DURATION_SEC = 60
MAX_TRIP_DURATION_SEC = 36_000
MIN_PASSENGERS = 1
MAX_PASSENGERS = 7

NYC_BOUNDS = {
    'min_lon': -74.5, 'max_lon': -73.3,
    'min_lat': 40.5, 'max_lat': 41.2,
}

SCATTER_SAMPLE_SIZE = 100_000
EARTH_RADIUS_KM = 6371
RANDOM_STATE = 42

# DBSCAN params
EPS_KM = 0.3
MIN_SAMPLES = 50
CLUSTERING_SAMPLE_SIZE = 50_000

In [ ]:
df = pd.read_csv(FILE_PATH)

print("--- Dataset Shape ---")
print(df.shape)
print("\n--- Missing Values ---")
print(df.isnull().sum())

In [ ]:
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df['dropoff_datetime'] = pd.to_datetime(df['dropoff_datetime'])

df['pickup_hour'] = df['pickup_datetime'].dt.hour
df['pickup_dayofweek'] = df['pickup_datetime'].dt.dayofweek
df['pickup_month'] = df['pickup_datetime'].dt.month

def haversine_distance(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return EARTH_RADIUS_KM * c

df['distance_km'] = haversine_distance(
    df['pickup_latitude'], df['pickup_longitude'],
    df['dropoff_latitude'], df['dropoff_longitude']
)

df.head()

In [ ]:
before = len(df)

df = df[df['trip_duration'].between(MIN_TRIP_DURATION_SEC, MAX_TRIP_DURATION_SEC)]
df = df[df['passenger_count'].between(MIN_PASSENGERS, MAX_PASSENGERS)]

df = df[
    df['pickup_longitude'].between(NYC_BOUNDS['min_lon'], NYC_BOUNDS['max_lon']) &
    df['pickup_latitude'].between(NYC_BOUNDS['min_lat'], NYC_BOUNDS['max_lat']) &
    df['dropoff_longitude'].between(NYC_BOUNDS['min_lon'], NYC_BOUNDS['max_lon']) &
    df['dropoff_latitude'].between(NYC_BOUNDS['min_lat'], NYC_BOUNDS['max_lat'])
]

print(f"Rows: {before:,} -> {len(df):,} ({before - len(df):,} dropped)")

In [ ]:
plt.figure(figsize=(10, 6))
df['log_trip_duration'] = np.log1p(df['trip_duration'])
sns.histplot(df['log_trip_duration'], bins=100, kde=True, color='blue')
plt.title('Distribution of Log-Transformed Trip Duration')
plt.xlabel('Log(Trip Duration + 1)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(x='pickup_hour', data=df, palette='viridis')
plt.title('Number of Trips by Hour of the Day')
plt.xlabel('Hour of the Day (0-23)')
plt.ylabel('Total Trips')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
avg_duration_day = df.groupby('pickup_dayofweek')['trip_duration'].mean().reset_index()
sns.barplot(x='pickup_dayofweek', y='trip_duration', data=avg_duration_day, palette='magma')
plt.title('Average Trip Duration by Day of the Week')
plt.xlabel('Day of the Week (0=Monday, 6=Sunday)')
plt.ylabel('Average Trip Duration (Seconds)')
plt.show()

In [ ]:
sample_df = df.sample(n=min(SCATTER_SAMPLE_SIZE, len(df)), random_state=RANDOM_STATE)

plt.figure(figsize=(10, 10))
plt.scatter(sample_df['pickup_longitude'], sample_df['pickup_latitude'], s=1, alpha=0.5, color='green')
plt.title('NYC Taxi Pickup Locations (Sampled)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.xlim(-74.05, -73.75)
plt.ylim(40.60, 40.90)
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
numeric_cols = ['passenger_count', 'trip_duration', 'pickup_hour', 'pickup_dayofweek', 'pickup_month', 'distance_km']
corr_matrix = df[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Matrix of Numerical Features')
plt.show()

In [ ]:
sample = df.sample(n=min(CLUSTERING_SAMPLE_SIZE, len(df)), random_state=RANDOM_STATE).copy()

coords_rad = np.radians(sample[['pickup_latitude', 'pickup_longitude']].values)
eps_rad = EPS_KM / EARTH_RADIUS_KM

db = DBSCAN(eps=eps_rad, min_samples=MIN_SAMPLES, metric='haversine', algorithm='ball_tree')
sample['zone'] = db.fit_predict(coords_rad)

n_zones = sample['zone'].nunique() - (1 if -1 in sample['zone'].values else 0)
n_noise = (sample['zone'] == -1).sum()
print(f"Found {n_zones} demand zones")
print(f"Noise points: {n_noise:,} / {len(sample):,} ({n_noise/len(sample):.1%})")

In [ ]:
plt.figure(figsize=(10, 10))

noise = sample[sample['zone'] == -1]
clustered = sample[sample['zone'] != -1]

plt.scatter(noise['pickup_longitude'], noise['pickup_latitude'],
            s=2, alpha=0.2, color='lightgray', label='noise')
plt.scatter(clustered['pickup_longitude'], clustered['pickup_latitude'],
            s=3, alpha=0.6, c=clustered['zone'], cmap='tab20')

plt.title(f'NYC Taxi Demand Zones (DBSCAN, eps={EPS_KM}km, min_samples={MIN_SAMPLES})')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.xlim(-74.05, -73.75)
plt.ylim(40.60, 40.90)
plt.legend()
plt.show()

In [ ]:
top_n = 8
clustered = sample[sample['zone'] != -1]
top_zones = clustered['zone'].value_counts().head(top_n).index
subset = clustered[clustered['zone'].isin(top_zones)]

pivot = (
    subset.groupby(['zone', 'pickup_hour'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=range(24), fill_value=0)
)

plt.figure(figsize=(14, 6))
sns.heatmap(pivot, cmap='YlOrRd', linewidths=0.3)
plt.title(f'Pickup Volume by Zone and Hour of Day (Top {top_n} Zones)')
plt.xlabel('Hour of Day')
plt.ylabel('Zone ID')
plt.show()